In [3]:
import os,sys 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append("../")       

import torch 
from torch import nn,optim
import torch.nn.functional as F 

from tqdm.notebook import tqdm
from drugsig.latent_dataset import LatentDataset
from torch.utils.data import DataLoader
from drugsig.predictor import CTP_Predictor,CTP_Predictor_resnet

from audtorch.metrics.functional import pearsonr

In [4]:
batch_size = 64
epochs = 300
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
k = 3

train_dataset = LatentDataset(latent_path=f"../data/drugsig/fold_{k}/train_latent.npy",rna_path=f"../data/drugsig/fold_{k}/train_df.csv")
train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,collate_fn=train_dataset.collate_fn,num_workers=8)

val_dataset = LatentDataset(latent_path=f"../data/drugsig/fold_{k}/val_latent.npy",rna_path=f"../data/drugsig/fold_{k}/val_df.csv")
val_dataloader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False,collate_fn=val_dataset.collate_fn,num_workers=8)

test_dataset = LatentDataset(latent_path=f"../data/drugsig/fold_{k}/test_latent.npy",rna_path=f"../data/drugsig/fold_{k}/test_df.csv")
test_dataloader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False,collate_fn=test_dataset.collate_fn,num_workers=8)

In [6]:
import numpy as np     
import random

def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
# 设置随机数种子

seed = 40
setup_seed(seed)

In [7]:
#model = Dleps_Predictor(latent_dim=64,dropout=0.35)
model = CTP_Predictor_resnet(latent_dim=64,output_dim=978,dropout=0.35)

In [8]:
model.to(device)

CTP_Predictor_resnet(
  (linear1): Linear(in_features=64, out_features=1024, bias=True)
  (acti1): ReLU()
  (drop1): Dropout(p=0.35, inplace=False)
  (block1): ResidueBlock(
    (linear): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.35, inplace=False)
  )
  (block2): ResidueBlock(
    (linear): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.35, inplace=False)
  )
  (block3): ResidueBlock(
    (linear): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.35, inplace=False)
  )
  (linear2): Linear(in_features=1024, out_features=1024, bias=True)
  (acti2): Tanh()
  (drop2): Dropout(p=0.35, inplace=False)
  (output): Linear(in_features=1024, out_features=978, bias=True)
)

In [9]:
adam_delta_optimizer = optim.Adadelta(model.parameters(),lr=1,rho=0.95,weight_decay=0.0)
#设置学习率调整 复现https://github.com/kekegg/DLEPS/blob/main/code/DLEPS/dleps_predictor.py
reduce_schedule = optim.lr_scheduler.ReduceLROnPlateau(optimizer=adam_delta_optimizer,mode="min",factor=0.2,patience=10,min_lr=1e-4)
total_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of parameters: {total_param}")

Total number of parameters: 5267410


In [10]:
train_loss = []
valid_loss = []
best_acc = 0
best_train_acc = 0.8
for epoch in tqdm(range(epochs)):
    
    model.train()
    #training phase 
    training_losses = 0
    #用于计算pearsons相关系数
    total_train_signatures = []
    total_train_preds = []
    training_iters = len(train_dataloader)
    for inputs in train_dataloader:
        smi_latents = inputs["latents"].to(device)
        gene_signatures = inputs["gene_signatures"].to(device) #[batch,978]
        total_train_signatures.append(gene_signatures)
        adam_delta_optimizer.zero_grad()
        y_preds = model(smi_latents) #[batch,978]
        total_train_preds.append(y_preds)
        loss = F.mse_loss(y_preds,gene_signatures,reduction="mean")
        loss.backward()
        adam_delta_optimizer.step()
        
        training_losses+=loss.item()
    mean_train_loss = training_losses / training_iters
    train_loss.append(mean_train_loss)
    
    #valid phase 
    validing_losses = 0 
    validing_iters = len(val_dataloader)
    with torch.no_grad():
        model.eval()
        
        for inputs in val_dataloader:
            smi_latents = inputs["latents"].to(device)
            gene_signatures = inputs["gene_signatures"].to(device) #[batch,978]
            
            y_preds = model(smi_latents) #[batch,978]
            loss = F.mse_loss(y_preds,gene_signatures,reduction="mean")
            validing_losses +=loss.item()
        
        mean_valid_loss = validing_losses / validing_iters
        valid_loss.append(mean_valid_loss)
        reduce_schedule.step(mean_valid_loss)
    
    #计算pearsons相关系数
    total_train_signatures = torch.cat(total_train_signatures,dim=0)
    total_train_preds = torch.cat(total_train_preds,dim=0)
    # print(f"train sig {total_train_signatures.shape} | train preds {total_train_preds.shape}")
    train_peasonrs = torch.mean(pearsonr(total_train_preds,total_train_signatures))
    #测试集相关系数
    total_val_signatures = []
    total_val_preds = []
    with torch.no_grad():
        model.eval()
        
        for inputs in val_dataloader:
            smi_latents = inputs["latents"].to(device)
            gene_signatures = inputs["gene_signatures"].to(device) #[batch,978]
            
            y_preds = model(smi_latents)
            total_val_signatures.append(gene_signatures)
            total_val_preds.append(y_preds)
            
        total_val_signatures = torch.cat(total_val_signatures,dim=0)
        total_val_preds = torch.cat(total_val_preds,dim=0)
        
        test_peasonrs = torch.mean(pearsonr(total_val_preds,total_val_signatures))
        # print(f"test sig {total_test_signatures.shape} | test preds {total_test_preds.shape}")
        
        # print(test_peasonrs)
        
    if test_peasonrs.item() > best_acc:
        best_acc = test_peasonrs.item()
        torch.save(model.state_dict(),f"../ckpts/drugsig/dleps_fold_{k}_seed_{seed}.ckpt")
        print(f" model saved \t| train_loss:{round(mean_train_loss,4)}\t| valid_loss:{round(mean_valid_loss,4)}\t | train_peasonrs:{round(train_peasonrs.item(),4)}\t |  val_peasonrs:{round(test_peasonrs.item(),4)}")
    
    if train_peasonrs.item() > best_train_acc:
        best_train_acc = train_peasonrs.item()
        print(f" training best \t| train_loss:{round(mean_train_loss,4)}\t| valid_loss:{round(mean_valid_loss,4)}\t | train_peasonrs:{round(train_peasonrs.item(),4)}\t | val_peasonrs:{round(test_peasonrs.item(),4)}")

  0%|          | 0/300 [00:00<?, ?it/s]

 model saved 	| train_loss:0.763	| valid_loss:0.6764	 | train_peasonrs:0.4032	 |  val_peasonrs:0.5299
 model saved 	| train_loss:0.6713	| valid_loss:0.6303	 | train_peasonrs:0.5074	 |  val_peasonrs:0.5402
 model saved 	| train_loss:0.6437	| valid_loss:0.6134	 | train_peasonrs:0.533	 |  val_peasonrs:0.5564
 model saved 	| train_loss:0.6083	| valid_loss:0.5853	 | train_peasonrs:0.5624	 |  val_peasonrs:0.593
 model saved 	| train_loss:0.5853	| valid_loss:0.5649	 | train_peasonrs:0.5805	 |  val_peasonrs:0.5977
 model saved 	| train_loss:0.5632	| valid_loss:0.5498	 | train_peasonrs:0.6002	 |  val_peasonrs:0.6227
 model saved 	| train_loss:0.5441	| valid_loss:0.5349	 | train_peasonrs:0.6136	 |  val_peasonrs:0.6282
 model saved 	| train_loss:0.5183	| valid_loss:0.5111	 | train_peasonrs:0.6318	 |  val_peasonrs:0.6331
 model saved 	| train_loss:0.5094	| valid_loss:0.503	 | train_peasonrs:0.6399	 |  val_peasonrs:0.6474
 model saved 	| train_loss:0.5036	| valid_loss:0.5005	 | train_peasonrs:0.644